In [ ]:
import re
import pandas as pd

SCREEN_FILE = r"alternative_splicing_or_splicing_high_relevance_references_dedup.xlsx"
# 1) Read data
ref_df = pd.read_excel(SCREEN_FILE)

# 2) Build regex pattern (targeted matching)
pattern = re.compile(
    r"\b(database|knowledge\s*base|knowledgebase|dataset|data\s*set|catalog(?:ue)?|biobank|atlas|portal|web\s*server|resources?)\b",
    flags=re.IGNORECASE,
)

# 3) Merge title + abstractText for screening
for col in ["title", "abstractText"]:
    if col not in ref_df.columns:
        ref_df[col] = ""

screen_text = (
    ref_df["title"].fillna("").astype(str)
    + " "
    + ref_df["abstractText"].fillna("").astype(str)
).str.strip()

# 4) Match and extract keywords
screened_df = ref_df.copy()
screened_df["screen_text"] = screen_text
screened_df["matched_terms"] = screened_df["screen_text"].apply(
    lambda x: sorted(set(m.lower() for m in pattern.findall(x)))
)
screened_df["has_database_terms"] = screened_df["matched_terms"].apply(lambda x: len(x) > 0)

# 5) Filter matched records
matched_df = screened_df[screened_df["has_database_terms"]].copy()

# 6) Statistics
total_n = len(screened_df)
matched_n = len(matched_df)
match_rate = matched_n / total_n if total_n else 0

term_freq = (
    matched_df["matched_terms"].explode().value_counts(dropna=True)
    if matched_n > 0
    else pd.Series(dtype="int64")
)

print(f"Total records: {total_n}")
print(f"Matched records: {matched_n}")
print(f"Match rate: {match_rate:.2%}")
print("Top matched terms:")
print(term_freq.head(20))

# Returned dataframe for downstream use
matched_df

Total records: 92130
Matched records: 6339
Match rate: 6.88%
Top matched terms:
matched_terms
database          2458
resource          1520
atlas             1178
resources          885
dataset            860
data set           366
catalog            228
portal             191
web server         183
catalogue          103
knowledgebase       65
biobank             56
webserver           44
knowledge base      37
Name: count, dtype: int64


,title,keywordList,web,pubYear,journal,citedByCount,pmid,pmcid,source,doi,abstractText,screen_text,matched_terms,has_database_terms
5,Basic local alignment search tool.,NaN,https://doi.org/10.1016/s0022-2836(05)80360-2,1990.0,Journal of molecular biology,62292,2231712.0,NaN,MED,10.1016/s0022-2836(05)80360-2,"A new approach to rapid sequence comparison, b...",Basic local alignment search tool. A new appro...,[database],True
7,Gapped BLAST and PSI-BLAST: a new generation o...,NaN,https://doi.org/10.1093/nar/25.17.3389,1997.0,Nucleic acids research,50787,9254694.0,PMC146917,MED,10.1093/nar/25.17.3389,The BLAST programs are widely used tools for s...,Gapped BLAST and PSI-BLAST: a new generation o...,[database],True
15,Gene set enrichment analysis: a knowledge-base...,NaN,https://doi.org/10.1073/pnas.0506580102,2005.0,Proceedings of the National Academy of Science...,41912,16199517.0,PMC1239896,MED,10.1073/pnas.0506580102,Although genomewide RNA expression analysis ha...,Gene set enrichment analysis: a knowledge-base...,[database],True
16,STAR: ultrafast universal RNA-seq aligner.,NaN,https://doi.org/10.1093/bioinformatics/bts635,2013.0,"Bioinformatics (Oxford, England)",41346,23104886.0,PMC3530905,MED,10.1093/bioinformatics/bts635,<h4>Motivation</h4>Accurate alignment of high-...,STAR: ultrafast universal RNA-seq aligner. <h4...,[dataset],True
32,Systematic and integrative analysis of large g...,NaN,https://doi.org/10.1038/nprot.2008.211,2009.0,Nature protocols,28537,19131956.0,NaN,MED,10.1038/nprot.2008.211,DAVID bioinformatics resources consists of an ...,Systematic and integrative analysis of large g...,"[knowledgebase, resources]",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92086,Jiawei yanghe decoction alleviates osteoporoti...,"{'keyword': ['Mesenchymal stem cell', 'Bone fo...",NaN,2025.0,Phytomedicine : international journal of phyto...,0,40929885.0,NaN,MED,10.1016/j.phymed.2025.157203,<h4>Background</h4>Osteoporotic osteoarthritis...,Jiawei yanghe decoction alleviates osteoporoti...,[database],True
92108,"ECD, a novel androgen receptor target promotes...",NaN,NaN,2025.0,Oncogene,0,40908313.0,PMC12518142,MED,10.1038/s41388-025-03559-x,Androgen receptor (AR)-mediated signaling is e...,"ECD, a novel androgen receptor target promotes...",[database],True
92113,Impact of alternative splicing on Arabidopsis ...,NaN,NaN,2025.0,bioRxiv : the preprint server for biology,0,38496481.0,PMC10942332,MED,10.1101/2024.02.29.582853,Limited proteomic evidence makes it unclear to...,Impact of alternative splicing on Arabidopsis ...,[dataset],True
92115,Universal features of alternative splicing and...,NaN,NaN,2025.0,TAG. Theoretical and applied genetics. Theoret...,0,40504371.0,NaN,MED,10.1007/s00122-025-04932-w,<h4>Key message</h4>The universal features of ...,Universal features of alternative splicing and...,"[dataset, resources]",True


: 

In [16]:
#AI check

import os
import re
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

# -----------------------------
# Configuration
# -----------------------------
OUTPUT_FILE = "AI_check.xlsx"
MODEL = "gpt-5.4"

# Ask only one question here
QUESTION = "Based on the provided title and abstract text and your knowledge, does this paper present or provide a database about splicing or alternative splicing?"

# Use environment variable for API key: OPENAI_API_KEY
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable is not set")

client = OpenAI(api_key=api_key)


# -----------------------------
# Helper function
# -----------------------------
def ask_yes_no(title: str, text: str, question: str, model: str = MODEL) -> str:
    """Return only Yes or No."""
    completion = client.chat.completions.create(
        model=model,
        max_completion_tokens=100,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a strict classifier. "
                    "You must answer with exactly one word: Yes or No. "
                ),
            },
            {
                "role": "user",
                "content": f"""
Title:
{title}

Text:
{text}

Question:
{question}

Output rule:
Answer exactly one word: Yes or No. Do not explain.
""".strip(),
            },
        ],
    )

    raw = (completion.choices[0].message.content or "").strip()
    normalized = re.sub(r"[^a-zA-Z]", "", raw).lower()

    # Normalize model output to strict Yes/No
    if re.search(r"\byes\b", raw, flags=re.IGNORECASE) or normalized.startswith("yes"):
        return "Yes"
    if re.search(r"\bno\b", raw, flags=re.IGNORECASE) or normalized.startswith("no"):
        return "No"

    # Safe fallback
    return "No"


# -----------------------------
# Main process
# -----------------------------
df = matched_df

if "title" not in df.columns:
    raise KeyError("Input file must contain a 'title' column.")

# Prefer abstractText, fallback to text
if "abstractText" in df.columns:
    text_col = "abstractText"
elif "text" in df.columns:
    text_col = "text"
else:
    raise KeyError("Input file must contain either 'abstractText' or 'text' column.")

results = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="AI Yes/No checking"):
    title = str(row.get("title", "") or "")
    text = str(row.get(text_col, "") or "")
    answer = ask_yes_no(title, text, QUESTION, model=MODEL)
    results.append(answer)

df["AI_check"] = results

df.to_excel(OUTPUT_FILE, index=False)
print(f"Done. Saved to: {OUTPUT_FILE}")
print(df[["title", "AI_check"]].head())


AI Yes/No checking: 100%|██████████| 4915/4915 [1:33:00<00:00,  1.14s/it]  


Done. Saved to: AI_check.xlsx
                                                title AI_check
5                  Basic local alignment search tool.       No
7   Gapped BLAST and PSI-BLAST: a new generation o...       No
15  Gene set enrichment analysis: a knowledge-base...       No
16         STAR: ultrafast universal RNA-seq aligner.       No
32  Systematic and integrative analysis of large g...       No


In [17]:
# Rebuild web column from existing AI_check.xlsx
import pandas as pd

FILE_PATH = "AI_check.xlsx"

df_ai = pd.read_excel(FILE_PATH)

if "doi" not in df_ai.columns:
    raise KeyError("AI_check.xlsx must contain a 'doi' column to rebuild 'web'.")

# Normalize DOI, then rebuild canonical DOI URL
doi_norm = (
    df_ai["doi"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.replace(r"^https?://(dx\.)?doi\.org/", "", regex=True)
)

df_ai["web"] = doi_norm.apply(lambda d: f"https://doi.org/{d}" if d else "")

df_ai.to_excel(FILE_PATH, index=False)
print(f"Done. Rebuilt 'web' from 'doi' and saved: {FILE_PATH}")
print(df_ai[["doi", "web"]].head())

Done. Rebuilt 'web' from 'doi' and saved: AI_check.xlsx
                             doi  \
0  10.1016/s0022-2836(05)80360-2   
1         10.1093/nar/25.17.3389   
2        10.1073/pnas.0506580102   
3  10.1093/bioinformatics/bts635   
4         10.1038/nprot.2008.211   

                                             web  
0  https://doi.org/10.1016/s0022-2836(05)80360-2  
1         https://doi.org/10.1093/nar/25.17.3389  
2        https://doi.org/10.1073/pnas.0506580102  
3  https://doi.org/10.1093/bioinformatics/bts635  
4         https://doi.org/10.1038/nprot.2008.211  
